In [ ]:
!pip install transformers torch


In [ ]:
import os
import json
import glob
import subprocess
import warnings
import torch
from transformers import pipeline
from transformers import logging as hf_logging
from tqdm import tqdm

# Ẩn các cảnh báo rác (như cảnh báo cắt ngang từ của Whisper)
warnings.filterwarnings("ignore")
hf_logging.set_verbosity_error()

# 1. Định nghĩa đường dẫn
media_dir = '/kaggle/input/your-media-dataset/' # ĐƯỜNG DẪN MẪU
save_path = '/kaggle/working/asr_results.json'

if not os.path.exists(media_dir):
    print(f'THÔNG BÁO: Thư mục {media_dir} không tồn tại. Vui lòng add data vào Kaggle qua nút "Add Input".')
    media_files = []
else:
    media_files = sorted(
        glob.glob(f"{media_dir}/**/*.mp4", recursive=True) + 
        glob.glob(f"{media_dir}/**/*.mp3", recursive=True) + 
        glob.glob(f"{media_dir}/**/*.wav", recursive=True)
    )

# 2. Khởi tạo Whisper
print('Loading model Whisper Large-v3 Turbo...')
device = "cuda:0" if torch.cuda.is_available() else "cpu"
pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-large-v3-turbo",
    device=device,
    chunk_length_s=30,
    return_timestamps=True,
)

results = []
print(f"Tìm thấy {len(media_files)} media files.")

for media_path in tqdm(media_files):
    video_id = os.path.splitext(os.path.basename(media_path))[0]
    is_video = media_path.lower().endswith('.mp4')
    target_audio_path = media_path
    
    if is_video:
        target_audio_path = f"/kaggle/working/temp_{video_id}.wav"
        cmd = ["ffmpeg", "-i", media_path, "-q:a", "0", "-map", "a", "-y", target_audio_path]
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        
        if not os.path.exists(target_audio_path):
            continue

    out = pipe(target_audio_path, generate_kwargs={"language": "vietnamese"})
    chunks = out.get("chunks", [])
    
    for chunk in chunks:
        ts = chunk.get("timestamp", (0.0, 0.0))
        if not isinstance(ts, (list, tuple)) or len(ts) != 2:
            ts = (0.0, 5.0)
            
        start_time, end_time = ts
        
        # SỬA LỖI: Cả start_time đôi khi cũng bị lỗi None từ model
        if start_time is None: 
            start_time = 0.0
        if end_time is None: 
            end_time = start_time + 5.0
        
        results.append({
            'video_id': video_id,
            'start_time': start_time,
            'end_time': end_time,
            'text': chunk.get("text", "").strip()
        })
        
    if is_video and os.path.exists(target_audio_path):
        os.remove(target_audio_path)

if results:
    with open(save_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f'Đã lưu kết quả tại {save_path}')
